# 02 — Ghép dữ liệu ngoài vào bảng thô

Ghép thêm cột **có sẵn nguồn** từ ERA5-Land, NASA POWER, CAMS và NASA FIRMS
vào `cleaned_data.csv`. Nguyên tắc:

1. Chỉ ghép cột, mọi thứ phải tính từ lịch sử để ở bước feature.
2. Cột trùng thông tin với dữ liệu gốc thì bỏ.
3. Cột quá yếu (importance thấp hoặc làm MAE tăng) thì bỏ.
4. Cột `pm2_5` của CAMS **phải bỏ**: nó là biến thể của target, gây data leakage.

**Đầu ra:** `data/processed/cleaned_data_v2.csv`

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = ROOT / "data" / "processed"
EXT = ROOT / "data" / "external"
OUT_DIR = ROOT / "results" / "data_quality"
TS = "Local Time"

ROOT, PROC, EXT

(WindowsPath('E:/Document/hoankiem-air-quality-forecasting'),
 WindowsPath('E:/Document/hoankiem-air-quality-forecasting/data/processed'),
 WindowsPath('E:/Document/hoankiem-air-quality-forecasting/data/external'))

## Bước 1 — Đọc dữ liệu gốc và bỏ cột không dùng

In [2]:
DROP_BASE = ["City", "Country Code", "Timezone", "AQI", "UV Index", "Solar Radiation"]

base = pd.read_csv(PROC / "cleaned_data.csv")
print("gốc:", base.shape)
base = base.drop(columns=[c for c in DROP_BASE if c in base.columns])
base[TS] = pd.to_datetime(base[TS])
base = base.set_index(TS).sort_index()
base.shape, list(base.columns)

gốc: (40608, 20)


((40608, 13),
 ['CO',
  'NO2',
  'O3',
  'PM10',
  'PM25',
  'SO2',
  'Temperature',
  'Relative Humidity',
  'Precipitation',
  'Clouds',
  'Pressure',
  'Wind Speed',
  'Wind Direction'])

## Bước 2 — Ghép ERA5-Land

Chỉ giữ 2 cột có thông tin **mới**: độ ẩm đất và chiều cao lớp kấy xáo.

In [3]:
era5land = pd.read_csv(EXT / "open_meteo" / "era5land_hoankiem_hourly.csv")
era5land["time"] = pd.to_datetime(era5land["time"])
era5land = era5land.set_index("time").sort_index()

ERA5LAND_KEEP = {
    "soil_moisture_0_to_7cm": "Soil Moisture",
    "boundary_layer_height": "Boundary Layer Height",
}
era5land_sel = era5land[list(ERA5LAND_KEEP)].rename(columns=ERA5LAND_KEEP)
print("bỏ (trùng dữ liệu gốc):", [c for c in era5land.columns if c not in ERA5LAND_KEEP])
era5land_sel.reindex(base.index).ffill(limit=3).describe().round(3)

bỏ (trùng dữ liệu gốc): ['source', 'surface_pressure', 'temperature_2m', 'wind_speed_10m', 'wind_direction_10m', 'shortwave_radiation', 'precipitation']


,Soil Moisture,Boundary Layer Height
count,40608.000,36243.000
mean,0.314,445.172
std,0.087,371.482
min,0.056,10.000
25%,0.261,145.000
50%,0.328,340.000
75%,0.384,655.000
max,0.442,2615.000


In [4]:
base = base.join(era5land_sel.reindex(base.index).ffill(limit=3), how="left")
base.shape

(40608, 15)

## Bước 3 — NASA POWER

Chỉ giữ `Dew Point` (điểm sương) và `Wind Speed 10m` (gió ở độ cao 10 m — dữ liệu gốc là 2 m).

In [5]:
power = pd.read_csv(EXT / "open_meteo" / "nasa_power_hoankiem_hourly.csv")
power["Local Time"] = pd.to_datetime(power["Local Time"])
power = power.set_index("Local Time").sort_index()

POWER_KEEP = {"T2MDEW": "Dew Point", "WS10M": "Wind Speed 10m"}
power_sel = power[list(POWER_KEEP)].rename(columns=POWER_KEEP)
print("bỏ (trùng dữ liệu gốc):", [c for c in power.columns if c not in POWER_KEEP and c not in ("key", "source", "year")])
power_sel.describe().round(2)

bỏ (trùng dữ liệu gốc): ['T2M', 'RH2M', 'WS2M', 'WD2M', 'PRECTOTCORR', 'PS', 'CLOUD_AMT']


,Dew Point,Wind Speed 10m
count,40608.00,40608.00
mean,20.58,2.65
std,5.86,1.35
min,-1.47,0.01
25%,16.85,1.74
50%,22.72,2.38
75%,25.41,3.32
max,28.63,17.20


In [6]:
base = base.join(power_sel.reindex(base.index).ffill(limit=3), how="left")
base.shape

(40608, 17)

## Bước 4 — NASA FIRMS (điểm cháy)

Dữ liệu cháy là **tổng hợp theo ngày**. Mỗi ngày có 1 con số, trải cho 24 giờ.
Chỉ giữ 3 cột có importance > 0.01 và không trùng nhau:

| Cột | r với PM2.5 | Importance | Giữ? |
|---|---|---|---|
| `fire_count_100km_roll3d` | +0.397 | 0.0104 | giữ |
| `fire_frp_100km_roll7d` | +0.333 | 0.0119 | giữ |
| `fire_dist_min_km` | −0.097 | 0.0107 | giữ |
| `fire_count_50km`, `fire_frp_50km` | +0.08 | < 0.005 | bỏ |
| `fire_count_100km`, `fire_frp_100km` | +0.3 | ~0.005 | bỏ (trùng bản 3 ngày/7 ngày) |
| `is_burn_season` | −0.206 | 0.0008 | bỏ (trùng `month_sin/cos`) |

In [7]:
fires = pd.read_csv(EXT / "firms" / "hanoi_fire_daily_features.csv")
fires["date"] = pd.to_datetime(fires["date"])
fires = fires.set_index("date").sort_index()

FIRMS_KEEP = {
    "fire_count_100km_roll3d": "Fire Count 100km 3d",
    "fire_frp_100km_roll7d": "Fire FRP 100km 7d",
    "fire_dist_min_km": "Fire Dist Min",
}
fires_sel = fires[list(FIRMS_KEEP)].rename(columns=FIRMS_KEEP)
fires_sel.head(3)

,Fire Count 100km 3d,Fire FRP 100km 7d,Fire Dist Min
date,,,
2022-01-13,0.0,0.0,999.000000
2022-01-14,0.0,0.0,183.623636
2022-01-15,0.0,0.0,192.461078


In [8]:
fire_hourly = fires_sel.reindex(base.index.normalize())
fire_hourly.index = base.index
base = base.join(fire_hourly, how="left")
base.shape

(40608, 20)

## Bước 5 — Hướng gió dạng tuần hoàn

`Wind Direction` là góc 0–360°. Dùng số thô thì model hiểu 359° và 1° là **rất xa nhau** — sai bản chất.
Chuyển sang 2 trục sin/cos để biểu diễn đúng tính tuần hoàn.

Đây là phép biến đổi **tất định từ 1 cột có sẵn**, không sinh thêm thông tin, nên đặt ở bước merge
để cả train lẫn inference dùng chung một định nghĩa.


In [9]:
wd = np.deg2rad(base["Wind Direction"])
base["wd_sin"] = np.sin(wd)
base["wd_cos"] = np.cos(wd)

base[["Wind Direction", "wd_sin", "wd_cos"]].tail(4).round(3)

,Wind Direction,wd_sin,wd_cos
Local Time,,,
2026-08-31 20:00:00,282.0,-0.978,0.208
2026-08-31 21:00:00,237.0,-0.839,-0.545
2026-08-31 22:00:00,283.0,-0.974,0.225
2026-08-31 23:00:00,300.0,-0.866,0.500


## Bước 6 — Kiểm tra và lưu

Bỏ cột `Dust` và `Aerosol Optical Depth` của CAMS: importance 0.0012 và 0.0148,
đồng thời thiếu 12% số giờ. Không đáng đưa vào vì sẽ phải xử lý NaN thêm.

In [10]:
report = {
    "rows": int(len(base)),
    "columns": int(base.shape[1]),
    "start": str(base.index.min()),
    "end": str(base.index.max()),
    "duplicate_timestamps": int(base.index.duplicated().sum()),
    "missing": {c: int(v) for c, v in base.isna().sum().items() if v > 0},
}
print(json.dumps(report, indent=2, ensure_ascii=False))

{
  "rows": 40608,
  "columns": 22,
  "start": "2022-01-13 00:00:00",
  "end": "2026-08-31 23:00:00",
  "duplicate_timestamps": 0,
  "missing": {
    "Boundary Layer Height": 4365
  }
}


In [11]:
print("Cột cuối cùng:")
for i, c in enumerate(base.columns, 1):
    print(f"  {i:>2}. {c}")

Cột cuối cùng:
   1. CO
   2. NO2
   3. O3
   4. PM10
   5. PM25
   6. SO2
   7. Temperature
   8. Relative Humidity
   9. Precipitation
  10. Clouds
  11. Pressure
  12. Wind Speed
  13. Wind Direction
  14. Soil Moisture
  15. Boundary Layer Height
  16. Dew Point
  17. Wind Speed 10m
  18. Fire Count 100km 3d
  19. Fire FRP 100km 7d
  20. Fire Dist Min
  21. wd_sin
  22. wd_cos


In [12]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
out = PROC / "cleaned_data_v2.csv"
base.reset_index().to_csv(out, index=False, encoding="utf-8")
(OUT_DIR / "merge_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("->", out, base.shape)

-> E:\Document\hoankiem-air-quality-forecasting\data\processed\cleaned_data_v2.csv (40608, 22)


## Bước 7 — Kiểm tra nhanh: cột mới có tín hiệu không?

Tương quan Pearson ở mức **3 giờ** (đã resample để bỏ chu kỳ ngày, tránh tương quan ảo).

In [13]:
d3 = base.resample("3h").mean(numeric_only=True)
table = []
for c in d3.columns:
    if c == "PM25":
        continue
    j = pd.concat([d3["PM25"], d3[c]], axis=1).dropna()
    table.append({"feature": c, "r": round(float(j.iloc[:, 0].corr(j.iloc[:, 1])), 3)})
pd.DataFrame(table).sort_values("r", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)

,feature,r
0,PM10,0.938
1,Soil Moisture,-0.456
2,Fire Count 100km 3d,0.397
3,Dew Point,-0.389
4,Pressure,0.363
5,Temperature,-0.337
6,Fire FRP 100km 7d,0.333
7,NO2,0.265
8,Wind Speed,-0.207
9,Wind Speed 10m,-0.189


**Kỳ vọng:** `Soil Moisture` (−0.46), `Dew Point` (−0.39), `Boundary Layer Height` (−0.18)
có tương quan âm rõ; `Fire Count 100km 3d` (+0.40) tương quan dương.

## Bước tiếp theo

`cleaned_data_v2.csv` là **raw đã ghép**, chưa có feature.

Sang batch feature engineering để tính: lag PM2.5, rolling, `PM10_PM25_ratio`,
`wd_sin/cos`, `hours_since_rain`, `Accumulated Hours of Rain`, `IsHoliday`.